# Cloud Data Warehouses — patterns with a local stand-in

All warehouse SQL patterns here run locally on DuckDB (columnar, like the real thing).
Each section notes the BigQuery equivalent so you can repeat it in the free sandbox.

## 1. Create an events table (locally; in BigQuery you'd use a public dataset)

In [ ]:
import duckdb
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)
n = 500_000
events = pd.DataFrame({
    "user_id": rng.integers(1, 5000, n),
    "event_date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "event_type": rng.choice(["view", "click", "purchase"], n, p=[0.7, 0.25, 0.05]),
    "revenue": np.where(rng.random(n) < 0.05, rng.uniform(5, 200, n).round(2), 0.0),
})
con = duckdb.connect()
con.register("events", events)
con.sql("SELECT event_type, COUNT(*) n FROM events GROUP BY 1 ORDER BY n DESC").df()

## 2. Window functions: running totals, deltas, dedup

In [ ]:
con.sql("""
WITH daily AS (
    SELECT event_date, SUM(revenue) AS revenue
    FROM events GROUP BY event_date
)
SELECT
    event_date,
    revenue,
    SUM(revenue)  OVER (ORDER BY event_date)                        AS running_total,
    revenue - LAG(revenue) OVER (ORDER BY event_date)               AS day_over_day,
    AVG(revenue)  OVER (ORDER BY event_date
                        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)   AS rolling_7d_avg
FROM daily
ORDER BY event_date
LIMIT 10
""").df()

In [ ]:
# The dedup / latest-per-key idiom — worth memorizing:
con.sql("""
SELECT * EXCLUDE (rn) FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_date DESC) AS rn
    FROM events
) WHERE rn = 1
LIMIT 5
""").df()

## 3. Partition pruning: write partitioned Parquet and compare scans

In [ ]:
import os
import time

con.execute("""
COPY (SELECT *, strftime(event_date, '%Y-%m') AS month FROM events)
TO '/tmp/events_partitioned' (FORMAT PARQUET, PARTITION_BY (month), OVERWRITE_OR_IGNORE)
""")

def timed(sql):
    t0 = time.time()
    result = con.execute(sql).fetchone()[0]
    return result, time.time() - t0

full, t_full = timed(
    "SELECT SUM(revenue) FROM read_parquet('/tmp/events_partitioned/**/*.parquet', hive_partitioning=1)")
pruned, t_pruned = timed(
    "SELECT SUM(revenue) FROM read_parquet('/tmp/events_partitioned/**/*.parquet', hive_partitioning=1) "
    "WHERE month = '2026-03'")

print(f"full scan:   {t_full*1000:.0f} ms")
print(f"pruned (1/6 months): {t_pruned*1000:.0f} ms")
# BigQuery equivalent: a table PARTITION BY DATE(event_date); the console shows
# bytes-billed dropping when you filter on the partition column.

## 4. 'Dry run': estimate before you execute

In [ ]:
# DuckDB: EXPLAIN shows the plan (files pruned). BigQuery: a dry run returns
# exact bytes scanned without running — shown (untested here) for reference:
print(con.execute(
    "EXPLAIN SELECT SUM(revenue) FROM read_parquet('/tmp/events_partitioned/**/*.parquet', "
    "hive_partitioning=1) WHERE month = '2026-03'").fetchall()[0][1][:600])

bigquery_dry_run = '''
from google.cloud import bigquery
client = bigquery.Client()
job = client.query("SELECT ...", job_config=bigquery.QueryJobConfig(dry_run=True))
print(f"would scan {job.total_bytes_processed / 1e9:.2f} GB")
'''

## 5. Mini-project

In the BigQuery sandbox, pick a `bigquery-public-data` dataset and record here:
- your analytical question and the window-function query answering it
- bytes scanned: naive query vs. partition-pruned + column-pruned version
- the cost difference at the on-demand price per TB